# COLMAP with Depth-Anything-V2

### How to use colmap

Full walkthrough: https://github.com/colmap/colmap?tab=readme-ov-file

Download Windows: https://github.com/colmap/colmap/releases


**Windows:**
1. Download COLMAP
2. Adjust the path in the code above for the "colmap.exe"-file
3. Look that the DepthAnythingV2 checkpoint is available and adjust the path below

### Pipeline

Setup all imports for the SfM/MVS pipeline

In [1]:
import torch
import subprocess
import os
import shutil
import struct
import open3d as o3d
from pathlib import Path
import numpy as np
from PIL import Image
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
from models.DepthAnythingV2.depth_anything_v2.dpt import DepthAnythingV2

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


xFormers not available
xFormers not available


Adjust paths for DepthAnythingV2 and COLMAP

In [2]:
DEPTHANYTHINGV2_PATH = r"./models/DepthAnythingV2/checkpoints/depth_anything_v2_vitl.pth"
COLMAP_PATH = r"C:\Program Files\colmap-x64-windows-cuda\bin\colmap.exe"

Load DepthAnythingV2 model checkpoint

In [3]:
model = DepthAnythingV2()
state_dict = torch.load(DEPTHANYTHINGV2_PATH, map_location="cpu")
model.load_state_dict(state_dict)
model.eval().cuda()

DepthAnythingV2(
  (pretrained): DinoVisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14))
      (norm): Identity()
    )
    (blocks): ModuleList(
      (0-23): 24 x NestedTensorBlock(
        (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
        (attn): MemEffAttention(
          (qkv): Linear(in_features=1024, out_features=3072, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=1024, out_features=1024, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ls1): LayerScale()
        (drop_path1): Identity()
        (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
        (mlp): Mlp(
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (act): GELU(approximate='none')
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
          (drop): Dropout(p=0.0, inplace=Fal

Setup paths for the SfM/MVS in- and output

In [4]:
colmap_dir = Path(r"./colmap")
images_dir = colmap_dir / "images//cabinet"
database_path = colmap_dir / "database.db"
sparse_output_path = colmap_dir / "sparse"
dense_output_path = colmap_dir / "dense"
dense_ply_output = colmap_dir / "dense_model.ply"

Prepare functions for the SfM

In [5]:
def run_colmap_cmd(*args):
    subprocess.run([COLMAP_PATH] + list(args), check=True)

def run_colmap_feature_extractor(db, imgs):
    run_colmap_cmd(
        "feature_extractor",
        "--database_path", db,
        "--image_path", imgs,
        "--ImageReader.camera_model", "PINHOLE",
        "--ImageReader.single_camera", "1",
        "--ImageReader.default_focal_length_factor", "1.2"
    )

def run_colmap_sequential_matcher(db, overlap=5):
    run_colmap_cmd(
        "sequential_matcher",
        "--database_path", db,
        "--SequentialMatching.overlap", str(overlap)
    )

def run_colmap_mapper(db, imgs, out):
    os.makedirs(out, exist_ok=True)
    run_colmap_cmd("mapper", "--database_path", db, "--image_path", imgs, "--output_path", out)

def run_colmap_model_converter(inp, out, typ="TXT"):
    run_colmap_cmd("model_converter", "--input_path", inp, "--output_path", out, "--output_type", typ)

def run_colmap_image_undistorter(imgs, sparse_model, dense_output):
    os.makedirs(dense_output, exist_ok=True)
    run_colmap_cmd(
        "image_undistorter",
        "--image_path", imgs,
        "--input_path", sparse_model,
        "--output_path", dense_output,
        "--output_type", "COLMAP",
        "--max_image_size", "2000"
    )

def run_colmap_patch_match_stereo(dense_output):
    """Run COLMAP's patch match stereo to generate depth and normal maps"""
    run_colmap_cmd(
        "patch_match_stereo",
        "--workspace_path", dense_output,
        "--workspace_format", "COLMAP",
        "--PatchMatchStereo.geom_consistency", "true"
    )

def run_colmap_stereo_fusion(dense_output, output_ply_path):
    run_colmap_cmd(
        "stereo_fusion",
        "--workspace_path", dense_output,
        "--workspace_format", "COLMAP",
        "--input_type", "geometric",
        "--output_path", output_ply_path
    )

Prepare functions for the MVS (with traditional COLMAP and DepthAnythingV2)

In [6]:
def save_colmap_depth_map(depth_array, output_path):
    height, width = depth_array.shape
    with open(output_path, 'wb') as f:
        header = f"{width}&{height}&{1}&"
        f.write(header.encode('utf-8'))
        depth_array.astype(np.float32).tofile(f)

def process_depth_with_depth_anything_v2():
    transform = Compose([
        Resize((518, 518)),
        ToTensor(),
        Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    input_dir = colmap_dir / "dense" / "images"
    ai_depth_maps_dir = colmap_dir / "dense" / "stereo" / "depth_maps_ai"
    os.makedirs(ai_depth_maps_dir, exist_ok=True)

    print("Generating depth maps with Depth Anything V2...")
    
    print("Pass 1: Generating depth maps...")
    depth_data = {}
    all_depth_values = []
    
    for img_name in os.listdir(input_dir):
        if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue

        print(f"Processing {img_name}...")
        img_path = input_dir / img_name
        img = Image.open(img_path).convert("RGB")
        original_size = img.size
        img_input = transform(img).unsqueeze(0).cuda()

        with torch.no_grad():
            depth = model(img_input)
            depth = depth.squeeze().cpu().numpy()

        depth_resized = np.array(Image.fromarray(depth).resize(original_size, Image.BILINEAR))
        
        depth_resized = np.max(depth_resized) - depth_resized

        depth_data[img_name] = {
            'depth': depth_resized,
            'original_size': original_size
        }
        
        all_depth_values.extend(depth_resized.flatten())

    if not depth_data:
        print("No valid images found!")
        return []

    global_min = np.min(all_depth_values)
    global_max = np.max(all_depth_values)
    
    print(f"Global depth range: {global_min:.4f} to {global_max:.4f}")
    
    if global_max <= global_min:
        print("ERROR: No depth variation found across images!")
        return []

    print("Pass 2: Normalizing and saving depth maps...")
    min_depth = 0.1
    max_depth = 20.0
    processed_images = []
    
    for img_name, data in depth_data.items():
        depth_resized = data['depth']
        
        depth_normalized = (depth_resized - global_min) / (global_max - global_min)

        depth_metric = min_depth + depth_normalized * (max_depth - min_depth)
        
        depth_metric = np.clip(depth_metric, min_depth, max_depth)
        
        valid_mask = (depth_metric > min_depth) & (depth_metric < max_depth)
        if not valid_mask.any():
            print(f"WARNING: No valid depths found for {img_name}")
            continue

        depth_filename = img_name.rsplit('.', 1)[0] + ".geometric.bin"
        depth_path = ai_depth_maps_dir / depth_filename
        save_colmap_depth_map(depth_metric, depth_path)

        print(f"Saved AI depth map: {depth_filename}")
        print(f"  Depth range: {depth_metric.min():.2f}m to {depth_metric.max():.2f}m")
        processed_images.append(img_name)

    print(f"Successfully processed {len(processed_images)} images")
    print(f"Global normalization: AI range [{global_min:.4f}, {global_max:.4f}] → Metric range [{min_depth}m, {max_depth}m]")
    
    return processed_images

def copy_ai_depths_to_stereo_folder():
    ai_depth_dir = colmap_dir / "dense" / "stereo" / "depth_maps_ai"
    colmap_depth_dir = colmap_dir / "dense" / "stereo" / "depth_maps"

    if not ai_depth_dir.exists():
        print("No AI depth maps found!")
        return

    os.makedirs(colmap_depth_dir, exist_ok=True)

    for depth_file in ai_depth_dir.glob("*.geometric.bin"):
        dest_file = colmap_depth_dir / depth_file.name
        shutil.copy2(depth_file, dest_file)
        print(f"Copied AI depth map: {depth_file.name}")

def run_stereo_fusion_with_ai_depths():
    print("Using AI-generated depth maps for fusion...")
    run_colmap_cmd(
        "stereo_fusion",
        "--workspace_path", str(dense_output_path),
        "--workspace_format", "COLMAP",
        "--input_type", "geometric",
        "--output_path", str(dense_ply_output),
        "--StereoFusion.min_num_pixels", "3",
        "--StereoFusion.max_reproj_error", "8.0",
        "--StereoFusion.max_depth_error", "0.2",
        "--StereoFusion.max_normal_error", "20",
        "--StereoFusion.check_num_images", "3"
    )

def debug_depth_maps():
    depth_dir = colmap_dir / "dense" / "stereo" / "depth_maps"
    if not depth_dir.exists():
        print("No depth maps directory found!")
        return

    for depth_file in depth_dir.glob("*.geometric.bin"):
        print(f"Checking {depth_file.name}...")
        try:
            with open(depth_file, 'rb') as f:
                header_data = f.read(100)
                header_str = header_data.decode('utf-8', errors='ignore')
                print(f"  Header: {header_str[:50]}...")
                header_end = header_str.rfind('&') + 1
                if header_end > 0:
                    dimensions = header_str[:header_end-1].split('&')
                    if len(dimensions) >= 3:
                        width, height, channels = map(int, dimensions[:3])
                        print(f"  Dimensions: {width}x{height}, channels: {channels}")
                        expected_size = header_end + width * height * 4
                        actual_size = depth_file.stat().st_size
                        print(f"  Expected size: {expected_size}, Actual size: {actual_size}")
                        if abs(expected_size - actual_size) > 100:
                            print(f"  WARNING: Size mismatch!")
                    else:
                        print(f"  ERROR: Invalid header format")
        except Exception as e:
            print(f"  ERROR reading file: {e}")

Run the whole next block to generate the sparse pointcloud and the AI supported dense cloud.

In [7]:
print("Running COLMAP sparse reconstruction...")
run_colmap_feature_extractor(str(database_path), str(images_dir))
run_colmap_sequential_matcher(str(database_path), overlap=3)
run_colmap_mapper(str(database_path), str(images_dir), str(sparse_output_path))

print("Converting sparse model to PLY...")
ply_output_path = colmap_dir / "sparse_model.ply"
run_colmap_model_converter(str(sparse_output_path / "0"), str(ply_output_path), typ="PLY")

print("Undistorting images...")
run_colmap_image_undistorter(str(images_dir), str(sparse_output_path / "0"), str(dense_output_path))

print("Generating depth maps with Depth Anything V2...")
processed_images = process_depth_with_depth_anything_v2()

print("Running COLMAP patch match stereo to generate normal maps...")
run_colmap_patch_match_stereo(str(dense_output_path))

print("Copying AI depth maps to stereo folder...")
copy_ai_depths_to_stereo_folder()

print("Debugging depth map format...")
debug_depth_maps()

print("Fusing AI depth maps into dense point cloud...")
run_stereo_fusion_with_ai_depths()

print(f"Dense reconstruction complete! Output saved to: {dense_ply_output}")
print("This point cloud was generated using Depth Anything V2 depth maps!")

Running COLMAP sparse reconstruction...
Converting sparse model to PLY...
Undistorting images...
Generating depth maps with Depth Anything V2...
Generating depth maps with Depth Anything V2...
Pass 1: Generating depth maps...
Processing 00100.jpg...
Processing 00259.jpg...
Processing 00263.jpg...
Processing 00268.jpg...
Processing 00281.jpg...
Processing 00284.jpg...
Processing 00285.jpg...
Processing 00292.jpg...
Processing 00297.jpg...
Processing 00303.jpg...
Processing 00307.jpg...
Processing 00312.jpg...
Processing 00447.jpg...
Processing 00457.jpg...
Processing 00458.jpg...
Processing 00463.jpg...
Processing 00468.jpg...
Global depth range: 0.0000 to 790.0547
Pass 2: Normalizing and saving depth maps...
Saved AI depth map: 00100.geometric.bin
  Depth range: 0.10m to 8.85m
Saved AI depth map: 00259.geometric.bin
  Depth range: 0.10m to 10.28m
Saved AI depth map: 00263.geometric.bin
  Depth range: 0.10m to 10.71m
Saved AI depth map: 00268.geometric.bin
  Depth range: 0.10m to 10.89m

Optional: Create a comparison by also running traditional COLMAP.

In [8]:
print("\n--- Creating comparison with traditional COLMAP ---")
traditional_output = colmap_dir / "dense_traditional.ply"
print("Running traditional COLMAP patch match stereo for comparison...")
run_colmap_patch_match_stereo(str(dense_output_path))
print("Fusing traditional COLMAP depth maps...")
run_colmap_cmd(
    "stereo_fusion",
    "--workspace_path", str(dense_output_path),
    "--workspace_format", "COLMAP", 
    "--input_type", "geometric",
    "--output_path", str(traditional_output)
)
print(f"Traditional COLMAP result saved to: {traditional_output}")
print("Now you can compare the AI-generated vs traditional results!")


--- Creating comparison with traditional COLMAP ---
Running traditional COLMAP patch match stereo for comparison...
Fusing traditional COLMAP depth maps...
Traditional COLMAP result saved to: colmap\dense_traditional.ply
Now you can compare the AI-generated vs traditional results!


### Filter Point Cloud

If the dense point cloud is too noisy, you can use the code above to remove some unneeded points.

In [ ]:
def filter_point_cloud(input_ply, output_ply):
    print(f"Filtering point cloud with gentle parameters: {input_ply}")
    pcd = o3d.io.read_point_cloud(str(input_ply))
    
    n_points_before = len(pcd.points)
    print(f"Points before filtering: {n_points_before}")
    
    print("Applying gentle statistical outlier removal...")
    pcd, _ = pcd.remove_statistical_outlier(
        nb_neighbors=50,    # ADJUST
        std_ratio=3.0       # ADJUST
    )
    
    print("Applying gentle radius outlier removal...")
    pcd, _ = pcd.remove_radius_outlier(
        nb_points=2,        # ADJUST
        radius=0.2          # ADJUST
    )
    
    n_points_after = len(pcd.points)
    print(f"Points after filtering: {n_points_after}")
    print(f"Removed {n_points_before - n_points_after} points ({(1 - n_points_after/n_points_before)*100:.1f}%)")
    
    o3d.io.write_point_cloud(str(output_ply), pcd)
    print(f"Saved filtered point cloud to: {output_ply}")
    
    return output_ply

Run the filtering on the dense point cloud.

In [ ]:
filtered_ply_output = colmap_dir / "dense_model_filtered.ply"
filter_point_cloud("C:\\Users\\fabia\\Desktop\\dense_model.ply", filtered_ply_output)

### Create Mesh

In [ ]:
import numpy as np
import open3d as o3d
from pathlib import Path
import struct

def read_colmap_points3d(points3d_path):
    points = []
    colors = []
    
    with open(points3d_path, 'rb') as f:
        num_points = struct.unpack('<Q', f.read(8))[0]
        
        for _ in range(num_points):
            point_id = struct.unpack('<Q', f.read(8))[0]
            x, y, z = struct.unpack('<ddd', f.read(24))
            points.append([x, y, z])
            r, g, b = struct.unpack('<BBB', f.read(3))
            colors.append([r/255.0, g/255.0, b/255.0])
            f.read(8)
            track_length = struct.unpack('<Q', f.read(8))[0]
            f.read(8 * track_length)
    
    return np.array(points), np.array(colors)

def read_colmap_ply(ply_path):
    return o3d.io.read_point_cloud(str(ply_path))

def create_mesh_from_colmap_pointcloud(input_path, output_path=None, 
                                     method='poisson', depth=9, 
                                     voxel_size=None, remove_outliers=True,
                                     estimate_normals=True, flip_normals=False,
                                     flip_triangles=True):
    input_path = Path(input_path)
    
    if input_path.suffix.lower() == '.ply':
        print("Reading PLY point cloud...")
        pcd = read_colmap_ply(input_path)
    elif input_path.name == 'points3D.bin':
        print("Reading COLMAP points3D.bin...")
        points, colors = read_colmap_points3d(input_path)
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(points)
        pcd.colors = o3d.utility.Vector3dVector(colors)
    else:
        raise ValueError("Unsupported file format. Use .ply or points3D.bin")
    
    print(f"Loaded point cloud with {len(pcd.points)} points")
    
    if voxel_size:
        print(f"Downsampling with voxel size {voxel_size}...")
        pcd = pcd.voxel_down_sample(voxel_size)
        print(f"After downsampling: {len(pcd.points)} points")
    
    if remove_outliers:
        print("Removing outliers...")
        pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
        print(f"After outlier removal: {len(pcd.points)} points")
    
    if estimate_normals:
        print("Estimating normals...")
        pcd.estimate_normals()
        pcd.orient_normals_consistent_tangent_plane(100)
        
        if flip_normals:
            print("Flipping normal directions...")
            pcd.normals = o3d.utility.Vector3dVector(-np.asarray(pcd.normals))
    
    print(f"Creating mesh using {method} method...")
    
    if method == 'poisson':
        mesh, _ = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
            pcd, depth=depth, width=0, scale=1.1, linear_fit=False
        )
        
        vertices_to_remove = np.asarray(mesh.vertices)[np.asarray(mesh.vertex_colors)[:, 0] < 0.1]
        if len(vertices_to_remove) > 0:
            print("Cleaning up low-density vertices...")
            mesh.remove_vertices_by_mask(np.asarray(mesh.vertex_colors)[:, 0] < 0.1)
    
    elif method == 'ball_pivoting':
        distances = pcd.compute_nearest_neighbor_distance()
        avg_dist = np.mean(distances)
        
        # Much denser ball pivoting with multiple radii
        radii = [
            avg_dist * 0.5,      # Very small balls for fine details
            avg_dist * 1.0,      # Small balls
            avg_dist * 1.5,      # Medium balls
            avg_dist * 2.0,      # Larger balls
            avg_dist * 3.0       # Fill remaining gaps
        ]
        
        mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(
            pcd, o3d.utility.DoubleVector(radii)
        )
    
    elif method == 'alpha_shape':
        mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_alpha_shape(
            pcd, alpha=0.03
        )
    
    else:
        raise ValueError("Method must be 'poisson', 'ball_pivoting', or 'alpha_shape'")
    
    print(f"Generated mesh with {len(mesh.vertices)} vertices and {len(mesh.triangles)} triangles")
    
    mesh.remove_degenerate_triangles()
    mesh.remove_duplicated_triangles()
    mesh.remove_duplicated_vertices()
    mesh.remove_non_manifold_edges()
    
    if flip_triangles:
        print("Flipping triangle winding order to fix inverted mesh...")
        triangles = np.asarray(mesh.triangles)
        mesh.triangles = o3d.utility.Vector3iVector(triangles[:, [0, 2, 1]])
    
    mesh = mesh.filter_smooth_simple(number_of_iterations=1)
    mesh.compute_vertex_normals()
    
    if output_path:
        print(f"Saving mesh to {output_path}...")
        o3d.io.write_triangle_mesh(str(output_path), mesh)
    
    return mesh

def visualize_mesh(mesh):
    print("Launching visualization...")
    o3d.visualization.draw_geometries([mesh])

if __name__ == "__main__":
    input_file = "C:/Users/fabia/Desktop/dav2_cabinet.ply"
    output_file = "./colmap/mesh_output.ply"
    method = 'poisson'  # Options: 'poisson', 'ball_pivoting', 'alpha_shape'

    if method == 'poisson':
        mesh = create_mesh_from_colmap_pointcloud(
                input_path=input_file,
                output_path=output_file,
                method='poisson',
                depth=12,
                voxel_size=0.01,
                remove_outliers=True,
                flip_triangles=False    # Change if mesh appears inverted
        )

    elif method == 'ball_pivoting':
        mesh = create_mesh_from_colmap_pointcloud(
            input_path=input_file,
            output_path=output_file,
            method='ball_pivoting',
            voxel_size=None,
            remove_outliers=False,
            flip_triangles=False
        )

    elif method == 'alpha_shape':
        mesh = create_mesh_from_colmap_pointcloud(
            input_path=input_file,
            output_path=output_file,
            method='alpha_shape',
            voxel_size=0.005,
            remove_outliers=True,
            flip_triangles=False
        )

    try:
        visualize_mesh(mesh)
    except FileNotFoundError:
        print("Please update the input_file path to your COLMAP point cloud")

In [ ]:
input_file = "C:/Github_FabianDubach/HSLU.DSPRO2.Beyond2D/colmap/dense_model.ply" # ADJUST
output_file = "./colmap/mesh_output.ply"
method = 'ball_pivoting'  # Options: 'poisson', 'ball_pivoting', 'alpha_shape'

if method == 'poisson':
    mesh = create_mesh_from_colmap_pointcloud(
            input_path=input_file,
            output_path=output_file,
            method='poisson',
            depth=12,
            voxel_size=None,
            remove_outliers=True,
            flip_triangles=False    # Change if mesh appears inverted
    )

elif method == 'ball_pivoting':
    mesh = create_mesh_from_colmap_pointcloud(
        input_path=input_file,
        output_path=output_file,
        method='ball_pivoting',
        voxel_size=None,
        remove_outliers=True,
        flip_triangles=True
    )

elif method == 'alpha_shape':
    mesh = create_mesh_from_colmap_pointcloud(
        input_path=input_file,
        output_path=output_file,
        method='alpha_shape',
        voxel_size=None,
        remove_outliers=True,
        flip_triangles=False
    )

try:
    visualize_mesh(mesh)
except FileNotFoundError:
    print("Please update the input_file path to your COLMAP point cloud")